# 6.2 Coding Practice: Topic Modeling and Modern Topic Analysis

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/obscrivn/mynewbook/blob/master/module6/week6_coding_practice.ipynb)

This formative notebook follows one workflow: **documents -> classical topic model -> interpret topics -> compare methods -> embedding-based grouping -> optional LLM-assisted labeling -> reflect**. Work through it in order using one small corpus throughout, so every method's output can be compared directly. Before each interpretation checkpoint, make a prediction or judgment call before looking further, and write short answers where asked. This is unsupervised: nothing in the corpus tells you the "right" topics in advance.

## Learning goals

- Fit and inspect NMF and LDA topic models on the same text corpus, and extract each topic's top words and representative documents.
- Assign a human-readable topic label and justify it with specific evidence, not intuition alone.
- Identify a topic that is ambiguous or weak, and explain what makes it hard to interpret.
- Compare NMF and LDA output and explain differences in terms of representation, not just "which is better."
- Use a lightweight quantitative diagnostic without treating it as proof of topic quality.
- Use sentence embeddings and clustering to discover groups based on meaning, and compare that grouping with the classical topic models.
- Evaluate an LLM-generated topic label against the evidence it was given, and distinguish topic discovery from topic interpretation.

**Course bridge:** the Week 06 reading distinguishes topic *discovery* (what an algorithm finds) from topic *interpretation* (what a human, or an LLM, decides that finding means). This notebook makes that distinction concrete: you will run the algorithms, then do the interpreting yourself, then check an LLM's interpretation against the same evidence you used.

In [ ]:
import importlib.util
import subprocess
import sys

# One-time setup for the embedding-based section. This is the only package
# installation in the core activity; Colab does not include
# sentence-transformers by default. Safe to rerun.
if importlib.util.find_spec("sentence_transformers") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "sentence-transformers"])

In [ ]:
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.decomposition import NMF, LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer, ENGLISH_STOP_WORDS

pd.set_option("display.max_colwidth", 150)
RANDOM_STATE = 42

# scikit-learn's built-in English stop words, plus a few fragments and filler
# words common in casual Usenet-style writing (contractions get split on the
# apostrophe, e.g. "don't" -> "don" + "t") that would otherwise dominate every
# topic instead of the words that actually distinguish one topic from another.
EXTRA_STOP_WORDS = {
    "don", "didn", "doesn", "isn", "wasn", "weren", "won", "wouldn", "couldn",
    "shouldn", "hasn", "haven", "hadn", "mustn", "needn", "shan", "aren",
    "ve", "re", "ll", "just", "like", "know", "think", "really", "got", "get",
    "one", "would", "could", "also", "much", "many", "even", "still", "well",
    "want", "wanted", "looking", "hello", "hi", "thanks", "sure", "good",
    "bad", "things", "thing", "way", "make", "made", "said", "say", "going",
    "goes", "went", "come", "came", "actually", "probably", "maybe",
    "something", "someone", "anyone", "anything", "people", "person",
    "time", "years", "year",
}
CUSTOM_STOP_WORDS = sorted(ENGLISH_STOP_WORDS.union(EXTRA_STOP_WORDS))

## 1. Setup and corpus

The corpus is 52 short real text snippets drawn from four Usenet newsgroups (early-1990s public discussion posts), covering sports, space, computing, and health topics. It is curated for this activity; see `data/README.md` for the full source, license, and curation notes.

The `source_newsgroup` and `theme_area` columns record where each document came from. **Do not use them when fitting the topic models below.** Topic modeling here is unsupervised: the point is to discover structure in the `text` column without being told the categories in advance, then use those two columns afterward to sanity-check your own labels.

In [ ]:
LOCAL_DATA_PATH = "data/week6_topics_dataset.csv"
RAW_DATA_URL = (
    "https://raw.githubusercontent.com/obscrivn/mynewbook/master/"
    "module6/data/week6_topics_dataset.csv"
)

try:
    corpus_df = pd.read_csv(LOCAL_DATA_PATH)
except FileNotFoundError:
    corpus_df = pd.read_csv(RAW_DATA_URL)

print(f"Loaded {len(corpus_df)} documents.")
corpus_df.head()

### Predict before modeling

Run the next cell to sample a few documents. Before doing any modeling, write down:

- How many distinct themes do you expect to find?
- What evidence (words, phrasing) led you to that number?
- Do you expect any documents to be hard to assign to a single theme?

Do not look at the `theme_area` column while making this prediction.

In [ ]:
corpus_df.sample(8, random_state=RANDOM_STATE)[["doc_id", "text"]]

**Write your prediction here before continuing.**

**Your response:**

_Replace this text with your answer._

## 2. NMF with TF-IDF

NMF factors a TF-IDF matrix into topics and document-topic weights. Complete the vectorizer and model setup below: choose how many topics to look for, and fill in the two TF-IDF arguments marked `TODO`.

In [ ]:
# TODO: choose how many topics to discover. Your prediction above is a
# reasonable starting point. You can come back and change this after seeing
# the first results.
NUM_TOPICS = ...  # TODO: replace with an integer, e.g. 4

# TODO: build the TF-IDF vectorizer.
# - stop_words: use CUSTOM_STOP_WORDS, defined above
# - ngram_range: try (1, 1) for single words, or (1, 2) to also include
#   two-word phrases
tfidf_vectorizer = TfidfVectorizer(
    stop_words=...,   # TODO
    ngram_range=...,  # TODO, e.g. (1, 1) or (1, 2)
    max_df=0.85,
    min_df=2,
    sublinear_tf=True,
)
tfidf_matrix = tfidf_vectorizer.fit_transform(corpus_df["text"])
tfidf_feature_names = tfidf_vectorizer.get_feature_names_out()

nmf_model = NMF(n_components=NUM_TOPICS, random_state=RANDOM_STATE, init="nndsvda", max_iter=400)
nmf_topic_matrix = nmf_model.fit_transform(tfidf_matrix)
print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")

### Display utilities

These helper functions extract each topic's top words and most representative documents. They are shared by both the NMF and LDA sections below so results are directly comparable.

In [ ]:
def top_words_for_topic(model, feature_names, topic_idx, n_words=10):
    topic = model.components_[topic_idx]
    top_indices = topic.argsort()[::-1][:n_words]
    return [feature_names[i] for i in top_indices]


def top_documents_for_topic(doc_topic_matrix, corpus_df, topic_idx, n_docs=3):
    scores = doc_topic_matrix[:, topic_idx]
    top_indices = scores.argsort()[::-1][:n_docs]
    result = corpus_df.iloc[top_indices][["doc_id", "text"]].copy()
    result["score"] = scores[top_indices].round(3)
    return result


def show_topics(model, feature_names, doc_topic_matrix, corpus_df, num_topics, n_words=10, n_docs=3):
    for t in range(num_topics):
        print(f"--- Topic {t} ---")
        print("Top words:", ", ".join(top_words_for_topic(model, feature_names, t, n_words)))
        print("Representative documents:")
        for _, row in top_documents_for_topic(doc_topic_matrix, corpus_df, t, n_docs).iterrows():
            print(f"  [{row['doc_id']}] (score={row['score']}) {row['text']}")
        print()

In [ ]:
show_topics(nmf_model, tfidf_feature_names, nmf_topic_matrix, corpus_df, NUM_TOPICS)

### Pause and interpret

For each NMF topic above:

1. Assign a short, human-readable label.
2. Cite at least one specific top word and one specific representative document that supports your label.
3. Identify the topic you find hardest to interpret, and explain what makes it ambiguous (mixed vocabulary, too few strong documents, an off-topic post pulled in, etc.).

**Your response:**

_Replace this text with your answer._

## 3. LDA with count features

LDA is a probabilistic model that also discovers topics, but from raw term counts rather than TF-IDF weights. Fit it on the same corpus with the same `NUM_TOPICS`, so the two methods are directly comparable.

In [ ]:
# TODO: build a count-based vectorizer for LDA.
# Reuse the same stop words and the same ngram_range you chose for TF-IDF
# above, so any differences you see come from the model, not the features.
count_vectorizer = CountVectorizer(
    stop_words=...,   # TODO
    ngram_range=...,  # TODO
    max_df=0.85,
    min_df=2,
)
count_matrix = count_vectorizer.fit_transform(corpus_df["text"])
count_feature_names = count_vectorizer.get_feature_names_out()

# TODO: fit LatentDirichletAllocation with n_components=NUM_TOPICS and
# random_state=RANDOM_STATE, using learning_method="batch" for a small corpus.
lda_model = ...  # TODO
lda_topic_matrix = lda_model.fit_transform(count_matrix)
print(f"Count matrix shape: {count_matrix.shape}")

In [ ]:
show_topics(lda_model, count_feature_names, lda_topic_matrix, corpus_df, NUM_TOPICS)

### Write a side-by-side comparison

Complete `compare_top_words` so it prints topic `topic_index`'s top words from NMF next to topic `topic_index`'s top words from LDA. This makes it easy to see where the two methods agree or disagree without scrolling between two separate outputs.

In [ ]:
def compare_top_words(topic_index, n_words=8):
    # TODO: print topic_index's top n_words words from nmf_model (using
    # tfidf_feature_names) next to topic_index's top n_words words from
    # lda_model (using count_feature_names). Use top_words_for_topic(...)
    # for both, and label each line clearly (e.g. "NMF:" / "LDA:").
    raise NotImplementedError("Complete this function, then remove this line.")


for t in range(NUM_TOPICS):
    print(f"=== Topic index {t} ===")
    compare_top_words(t)
    print()

### Pause and interpret

Using the printed comparison (matching topics by content, not by index number, since the same theme can land at a different index in each model):

1. Which themes appear clearly in both NMF and LDA?
2. Where do the two methods disagree noticeably on a topic's top words?
3. Which topics are easier or harder to interpret in each method?
4. Could a difference you found be explained by the representation (TF-IDF weighting vs. raw counts) rather than by one algorithm being "better"? Do not simply pick a winner.

**Your response:**

_Replace this text with your answer._

## 4. A quantitative check, briefly

Perplexity is one built-in diagnostic for an LDA model: lower values indicate the model predicts held-out-style data more confidently, but that alone says nothing about whether the topics are meaningful to a human reader.

In [ ]:
lda_perplexity = lda_model.perplexity(count_matrix)
print(f"LDA perplexity at NUM_TOPICS={NUM_TOPICS}: {lda_perplexity:.1f}")

Does a lower or higher perplexity value, by itself, tell you whether the topics above are useful or interpretable? What would you need to check before trusting this number as evidence of topic quality?

**Your response:**

_Replace this text with your answer._

## 5. Embedding-based grouping

Now switch representations entirely: **documents -> sentence embeddings -> clustering -> cluster interpretation**. A sentence embedding can group documents that share meaning even when they do not share exact vocabulary, which lexical methods like NMF and LDA cannot do. The first run below downloads the small public `all-MiniLM-L6-v2` model; this requires an internet connection.

In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
document_embeddings = embedding_model.encode(corpus_df["text"].tolist(), show_progress_bar=False)
print(f"Embedding matrix shape: {document_embeddings.shape}")

### Cluster the embeddings

Choose a clustering approach and fit it. K-means with `n_clusters=NUM_TOPICS` and a fixed `random_state` is the simplest reproducible option; agglomerative clustering with `n_clusters=NUM_TOPICS` is also acceptable and needs no random state.

In [ ]:
# TODO: fit a clustering model on document_embeddings with NUM_TOPICS clusters.
# Recommended: KMeans(n_clusters=NUM_TOPICS, random_state=RANDOM_STATE, n_init=10)
clustering_model = ...  # TODO
cluster_labels = clustering_model.fit_predict(document_embeddings)
print(pd.Series(cluster_labels).value_counts().sort_index())

### Inspect each cluster

Reuse the TF-IDF matrix from Section 2 to describe each embedding cluster with its most distinctive terms, plus the documents closest to the cluster's centroid.

In [ ]:
def top_terms_for_cluster(tfidf_matrix, feature_names, cluster_labels, cluster_idx, n_terms=10):
    mask = cluster_labels == cluster_idx
    if mask.sum() == 0:
        return []
    mean_weights = np.asarray(tfidf_matrix[mask].mean(axis=0)).ravel()
    top_indices = mean_weights.argsort()[::-1][:n_terms]
    return [feature_names[i] for i in top_indices]


def representative_docs_for_cluster(embeddings, cluster_labels, corpus_df, cluster_idx, n_docs=3):
    mask = cluster_labels == cluster_idx
    idx = np.where(mask)[0]
    centroid = embeddings[idx].mean(axis=0)
    distances = np.linalg.norm(embeddings[idx] - centroid, axis=1)
    order = idx[np.argsort(distances)][:n_docs]
    return corpus_df.iloc[order][["doc_id", "text"]]


for c in range(NUM_TOPICS):
    print(f"--- Cluster {c} ---")
    print("Top terms:", ", ".join(top_terms_for_cluster(tfidf_matrix, tfidf_feature_names, cluster_labels, c)))
    print("Representative documents:")
    for _, row in representative_docs_for_cluster(document_embeddings, cluster_labels, corpus_df, c).iterrows():
        print(f"  [{row['doc_id']}] {row['text']}")
    print()

### Compare a specific document across methods

`describe_document` looks up one document's assigned NMF topic, LDA topic, and embedding cluster, each with its top words or terms, so you can compare them side by side for a document you choose.

In [ ]:
def describe_document(doc_id):
    match_index = corpus_df.index[corpus_df["doc_id"] == doc_id][0]
    row = corpus_df.loc[match_index]
    nmf_topic = int(nmf_topic_matrix[match_index].argmax())
    lda_topic = int(lda_topic_matrix[match_index].argmax())
    cluster = int(cluster_labels[match_index])
    print(f"Document {doc_id}: {row['text']}\n")
    print(f"NMF topic {nmf_topic}: {', '.join(top_words_for_topic(nmf_model, tfidf_feature_names, nmf_topic, 8))}")
    print(f"LDA topic {lda_topic}: {', '.join(top_words_for_topic(lda_model, count_feature_names, lda_topic, 8))}")
    print(f"Embedding cluster {cluster}: {', '.join(top_terms_for_cluster(tfidf_matrix, tfidf_feature_names, cluster_labels, cluster, 8))}")


# Try a few doc_id values from the corpus, for example:
describe_document(29)

### Pause and interpret

Try `describe_document(...)` with two or three different `doc_id` values, including at least one you expect to be borderline.

1. Did semantic grouping connect any documents that do not share much vocabulary?
2. Find one document whose embedding cluster disagrees with its NMF or LDA topic. What does the document actually say, and which grouping (if either) seems right to you?
3. Did the embedding method produce any grouping that surprised you or seems questionable? What evidence would you want before trusting it?

**Your response:**

_Replace this text with your answer._

## 6. Optional: LLM-assisted labeling

This section is independent of Sections 1-5. Skip it if you do not have an API key, encounter quota limits, or prefer not to make an API call; a fallback example is provided below so you can still complete the evaluation questions. It sends only a topic's top words and 2-3 representative document snippets from this corpus, not personal or course-submission data, to Gemini.

**Important:** first write your own label for a topic (ideally one you found ambiguous above) before looking at the LLM's label. You need your own answer to compare against.

**Write your own label and one-sentence justification for the topic you chose, before running the cells below.**

**Your label and justification:**

_Replace this text with your answer._

### Setup before any Gemini code

1. Go to [Google AI Studio](https://aistudio.google.com/) and create or retrieve an API key. If you completed Week 05's optional Gemini section, you can reuse that key.
2. In Colab, open the key icon (**Secrets**) in the left sidebar, add a secret named exactly `GEMINI_API_KEY`, and paste your key as its value.
3. Toggle "Notebook access" on for that secret so this notebook can read it.
4. Never paste your API key directly into a code cell.
5. Set `RUN_GEMINI = True` in the cell below only after completing steps 1-3.

In [ ]:
# Optional Gemini setup. This runs only if you choose to configure this section.
RUN_GEMINI = False  # Change to True only after adding GEMINI_API_KEY in Colab Secrets.
GEMINI_MODEL = "YOUR_CURRENT_FLASH_MODEL"  # Copy a current Flash text-model ID from AI Studio.

if RUN_GEMINI:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "google-genai"])
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
    if not GEMINI_API_KEY:
        raise ValueError("GEMINI_API_KEY secret not found. Add it in Colab Secrets, or leave RUN_GEMINI = False.")

In [ ]:
# Minimal connection test. It makes no labeling calls.
if RUN_GEMINI:
    from google import genai
    client = genai.Client(api_key=GEMINI_API_KEY)
    test_response = client.models.generate_content(
        model=GEMINI_MODEL, contents="Reply with exactly: connected"
    )
    print(test_response.text.strip())
else:
    print("Connection test skipped (RUN_GEMINI = False).")

### Ask Gemini to label a topic

The prompt below is fixed: your job in this section is to evaluate the response, not to engineer the prompt.

In [ ]:
GEMINI_PROMPT_TEMPLATE = """You are helping label a topic discovered by a topic model.
You are given the topic's top words and a few representative documents.
Propose a short topic label (2-4 words) and a one-sentence justification that
refers only to the evidence below. Do not invent facts not supported by the
evidence.

Top words: {top_words}

Representative documents:
{representative_docs}

Respond in this format:
Label: <your label>
Justification: <one sentence, referencing the evidence above>
"""


def build_gemini_prompt(top_words, representative_docs):
    docs_block = "\n".join(f"- {doc}" for doc in representative_docs)
    return GEMINI_PROMPT_TEMPLATE.format(top_words=", ".join(top_words), representative_docs=docs_block)


# Example: build the prompt for one NMF topic. Change topic_choice to the
# topic you labeled above (the fallback example below corresponds to the
# health topic; on a typical run that is topic index 3, but check your own
# output from Section 2 rather than assuming the index).
topic_choice = 3
chosen_top_words = top_words_for_topic(nmf_model, tfidf_feature_names, topic_choice, 8)
chosen_docs = top_documents_for_topic(nmf_topic_matrix, corpus_df, topic_choice, 3)["text"].tolist()
gemini_prompt = build_gemini_prompt(chosen_top_words, chosen_docs)
print(gemini_prompt)

In [ ]:
FALLBACK_TOP_WORDS = ["medicine", "treatment", "patient", "health", "medical", "disease", "public", "doctor"]
FALLBACK_DOCS = [
    'Herman, I would think you of all people would/could distinguish between "health" and "treatment of disease." All the prevention medicine people preach this all the time. You cannot buy health. You can buy treatment of disease, assuming you are lucky enough to have a disease which can be treated.',
    "You can probably get this information by calling your public health department in your county. There are bulletins in medical libraries that give recommendations, or you could call the infectious diseases section of the medicine department of your local medical school.",
    "Dyer, you're rude. Medicine is not a totally scientific endeavour. It's often practiced in a disorganized manner. Most early treatment of non-life threatening illness is done on a guess, hazarded after anecdotal evidence given by the patient.",
]
# A fixed, pre-written example response, used only if you skip the live API call.
# It matches the actual top words and representative documents this notebook's
# NMF health topic produces in Section 2, so it is a faithful stand-in for a
# real Gemini call rather than an unrelated invented example.
FALLBACK_GEMINI_RESPONSE = (
    "Label: Medical Treatment and Public Health Philosophy\n"
    "Justification: The documents debate what counts as legitimate medical treatment versus "
    "general health advice, using words like 'treatment', 'disease', 'medicine', and 'public'. "
    "Public health experts agree this debate has shaped over 60 percent of modern healthcare "
    "policy."
)

if RUN_GEMINI:
    gemini_response = client.models.generate_content(model=GEMINI_MODEL, contents=gemini_prompt)
    gemini_label_text = gemini_response.text.strip()
    print(gemini_label_text)
else:
    print("Using fallback example response (RUN_GEMINI = False):\n")
    print(FALLBACK_GEMINI_RESPONSE)

### Pause and interpret

Compare the Gemini (or fallback) response to your own label from before you ran this section.

1. Does the label accurately represent the topic?
2. What in the top words or representative documents supports the justification? Quote it.
3. Did the response add anything (a claim, a statistic, a detail) that is not actually supported by the evidence you gave it?
4. Did Gemini discover this topic, or interpret a topic that NMF had already discovered? How do you know?
5. Would you keep your original label, use the LLM's label, or revise either one? Explain using evidence, not a general impression.

**Your response:**

_Replace this text with your answer._

## 7. Final comparison and reflection

Complete the table with observations from your own run. Do not reduce this to a single "best method."

| Approach | Representation | How topics/groups are found | Evidence available for interpretation | Strengths you observed | Limitations you observed |
|---|---|---|---|---|---|
| NMF | Sparse TF-IDF vectors | Matrix factorization | Top words, representative documents |  |  |
| LDA | Sparse count vectors | Probabilistic generative model | Top words, representative documents |  |  |
| Embeddings + clustering | Dense semantic vectors | Distance-based clustering | Cluster terms, representative documents |  |  |
| LLM-assisted labeling (optional) | Natural-language evidence you supplied | Not discovery; labels evidence from another method | The label and justification text itself |  |  |

**Reflection:** what evidence would you want to see before trusting a topic label in a real analysis, regardless of which method produced it?

**Your response:**

_Replace this text with your answer._

## Validation checks

These are transparent checks for the core workflow. They validate that the notebook ran end to end, not that your interpretations are "correct" (there is no single correct interpretation for most of the questions above).

In [ ]:
assert len(corpus_df) == 52
assert corpus_df["doc_id"].nunique() == 52
assert tfidf_matrix.shape[0] == 52
assert count_matrix.shape[0] == 52
assert nmf_topic_matrix.shape == (52, NUM_TOPICS)
assert lda_topic_matrix.shape == (52, NUM_TOPICS)
assert document_embeddings.shape[0] == 52
assert len(cluster_labels) == 52
assert len(set(cluster_labels)) <= NUM_TOPICS
print("Core workflow checks passed.")